# 5장 — 셀프 어텐션: 어디를 볼지 내용으로 정한다 (실습)

교재 `docs/book/05-attention.md` 와 함께 본다. 이 노트북에서 하는 것:

1. "과거의 평균" 을 행렬곱 하나로 쓰는 트릭 — 어텐션의 뼈대
2. Q·K·V 와 softmax(QKᵀ/√d)V 를 한 줄씩, 그리고 `attention()` 함수로
3. 인과 마스크, 멀티헤드, 어텐션 맵 그림
4. 어텐션 층 하나를 얹은 모델이 4장 MLP 보다 나은지 (짧은 학습)

> 전체 실행 약 2분.

## 1. 과거의 평균 — 가장 단순한 '문맥 섞기'

In [ ]:
import math
import time

import torch
import torch.nn.functional as F

from shllm.config import TOKENIZER_DIR, setup_cpu

setup_cpu()
B, T, C = 1, 5, 3
x = torch.arange(T * C, dtype=torch.float32).view(B, T, C)  # 눈으로 보기 쉬운 값
print("x[0] =\n", x[0])

# 자리 t 의 출력 = 자리 0..t 의 평균. 하삼각 행렬을 행 정규화하면 "평균 가중치" 가 된다
tril = torch.tril(torch.ones(T, T))
W = tril / tril.sum(dim=1, keepdim=True)  # (T, T)
print("가중치 W =\n", W)
print("W @ x[0] =\n", W @ x[0])  # 행 t = 0..t 평균

`(T, T)` 행렬 하나가 "누가 누구를 얼마나 보는가"를 정한다. 지금은 균등 평균이라 자리만 보고 정했다.
**어텐션은 이 가중치를 데이터에서 계산한다** — 어떤 토큰이 어떤 토큰을 더 볼지, 내용(벡터)으로.

## 2. Q · K · V

In [ ]:
torch.manual_seed(1)
B, T, C, d = 1, 5, 8, 4
x = torch.randn(B, T, C)
Wq, Wk, Wv = (torch.randn(C, d) * 0.5 for _ in range(3))
q, k, v = x @ Wq, x @ Wk, x @ Wv  # 각 (B, T, d): 질문 · 꼬리표 · 내용

scores = q @ k.transpose(-2, -1) / math.sqrt(d)  # (B, T, T)  자리 i 의 질문 · 자리 j 의 꼬리표
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
scores = scores.masked_fill(~mask, float("-inf"))  # 미래는 -inf → softmax 후 0
weights = F.softmax(scores, dim=-1)
out = weights @ v  # (B, T, d)
print("weights[0] =\n", weights[0].numpy().round(2))
print("out", tuple(out.shape))

In [ ]:
from shllm.attention import attention, causal_mask

out2, w2 = attention(q, k, v, causal_mask(T))
print("모듈과 같은가:", torch.allclose(out, out2), torch.allclose(weights, w2))

자리 i 의 출력은 "i 이하의 자리 중 질문과 꼬리표가 맞는 곳의 내용을 섞은 것". 세 투영이 각각 하는 일:

| | 뜻 | 자바 비유 |
|---|---|---|
| Q (query) | 내가 찾는 것 | `Map.get(key)` 의 key |
| K (key) | 나를 찾을 때 쓸 꼬리표 | 맵의 저장된 key |
| V (value) | 찾아지면 넘겨줄 내용 | 맵의 value |

다른 점: 정확히 일치하는 key 하나가 아니라 **모든 key 와의 유사도로 부드럽게** 섞는다(soft lookup).

### 2.1 √d 로 나누는 이유

In [ ]:
torch.manual_seed(0)
for dd in (4, 64, 512):
    qq, kk = torch.randn(1000, dd), torch.randn(1000, dd)
    raw = (qq * kk).sum(-1)
    print(f"d={dd:>3}: 내적의 표준편차 {raw.std():.1f}  → /√d 하면 {(raw / math.sqrt(dd)).std():.2f}")

d 가 크면 내적 값이 √d 에 비례해 커지고, softmax 가 한 곳에 0.999 로 몰려 기울기가 사라진다. √d 로 나누면 어느 d 에서든 분산 1 근처.

## 3. 마스크 · 멀티헤드 · 어텐션 맵

In [ ]:
from shllm.attention import CausalSelfAttention
from shllm.data import load_corpus
from shllm.embedding import GPTEmbedding
from shllm.tokenizer import BPETokenizer

tok = BPETokenizer.load(TOKENIZER_DIR / "bpe-8192.json")
text = load_corpus("korean-classics")
data = torch.tensor(tok.encode(text))
n = int(0.9 * len(data))
train, val = data[:n], data[n:]

torch.manual_seed(0)
C, H = 64, 4
emb = GPTEmbedding(tok.vocab_size, block_size=32, n_embd=C)
attn = CausalSelfAttention(n_embd=C, n_head=H, block_size=32)
idx = data[None, :12]
y = attn(emb(idx))
print("입력", tuple(idx.shape), "→ 임베딩", tuple(emb(idx).shape), "→ 어텐션", tuple(y.shape))
print("어텐션 가중치", tuple(attn.last_weights.shape), "= (B, 헤드, T, T)")
print("파라미터:", sum(p.numel() for p in attn.parameters()), "= qkv 3C²+3C + proj C²+C")

초기화 직후라 가중치는 대략 균등이다. 학습된 모델의 맵은 6·7장 이후에 본다. 여기서는 **모양**을 익힌다.

## 4. 어텐션이 정말 나은가 — 4장 MLP 와 같은 조건으로

문맥 8 토큰, 임베딩 64, 학습 1,500 스텝. MLP 는 문맥을 이어 붙여 `hidden` 에 넣었고, 여기서는 어텐션 층 하나 + 같은 은닉층을 쓴다.

In [ ]:
import torch.nn as nn

from shllm.data import get_batch
from shllm.mlp import MLPLanguageModel, train_steps


class TinyAttentionLM(nn.Module):
    """임베딩 → 어텐션 한 층 → 은닉층 → 어휘. 6장 GPT 의 축소판 (LayerNorm·잔차 없음)."""

    def __init__(self, V, T, C, H, hidden):
        super().__init__()
        self.emb = GPTEmbedding(V, T, C)
        self.attn = CausalSelfAttention(C, H, T)
        self.hidden = nn.Linear(C, hidden)
        self.out = nn.Linear(hidden, V)

    def forward(self, idx, targets=None):
        x = self.attn(self.emb(idx))  # (B, T, C)
        logits = self.out(torch.tanh(self.hidden(x)))  # (B, T, V) — 모든 자리에서 예측
        loss = None if targets is None else F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss


T = 8
results = {}
for name, model, last_only in (
    ("MLP T=8", MLPLanguageModel(tok.vocab_size, T, 64, 256), True),
    ("Attention T=8", TinyAttentionLM(tok.vocab_size, T, 64, 4, 256), False),
):
    torch.manual_seed(0)
    g = torch.Generator().manual_seed(0)

    def batch():
        bx, by = get_batch(train, T, 64, g)
        return (bx, by[:, -1]) if last_only else (bx, by)

    t0 = time.perf_counter()
    losses = train_steps(model, batch, steps=1500, lr=3e-3, log_every=0)
    with torch.no_grad():
        vx, vy = get_batch(val, T, 2048, torch.Generator().manual_seed(1))
        logits, _ = model(vx, vy[:, -1] if last_only else vy)
        # 공정 비교: 두 모델 다 마지막 자리의 다음 토큰 loss 로
        last_logits = logits if last_only else logits[:, -1]
        vl = F.cross_entropy(last_logits, vy[:, -1]).item()
    results[name] = vl
    print(f"{name:<16} 파라미터 {sum(p.numel() for p in model.parameters()):>9,}  {time.perf_counter() - t0:.0f}초  val(마지막 자리) {vl:.3f}")

어텐션 모델은 T 개 자리 **모두**에서 다음 토큰을 예측하므로 같은 배치에서 8배 많은 학습 신호를 얻고, 문맥 어디에 있든 같은 가중치로 토큰을 본다.
그 결과 같은 스텝 수에서 val loss 가 더 낮다(수치는 셀 출력). 6장에서 여기에 LayerNorm·잔차·MLP 를 더해 블록을 만들고 쌓는다.

## 정리

- 어텐션 = `(T, T)` 시선 가중치를 데이터로 계산해 값(V)을 섞는 것. `softmax(QKᵀ/√d)V` 한 줄.
- 인과 마스크로 미래를 가리고, √d 로 softmax 포화를 막고, 여러 헤드로 여러 관계를 동시에 본다.
- 어텐션 자체는 순서를 모른다 → 3장 위치 임베딩이 입력에 있어야 한다.

---
**다음 장**: 6장 — Transformer 블록을 만들고 쌓아 GPT 를 조립한다.